# LD comparison — SNPs (GrENE-Net vs production) and SVs (production)

Two questions:

**(A) Are SNP LD-decay curves comparable between two 231-founder panels?**
- GrENE-Net 231-founder SNP-only catalog (`data/vcf/greneNet_final_v1.1.recode.vcf.gz`)
- Our production merged 231-founder pangenome (`pangenie_genotyping/data/merged/founders_231_chr.vcf.gz`) — SNP records only

Same panel size + same ecotypes → SNP-LD curves SHOULD overlay if both catalogs capture the same population structure.

**(B) How does SV LD compare to SNP LD in the production panel?**
- SV anchors (50bp+) in `founders_231_chr.vcf.gz`
- Compared with their SNP/indel/SV neighbors within 1Mb
- Tells us how taggable SVs are by SNPs (relevant for downstream GWAS / haplotype methods)

## Method

All LD r² computed via `preprocess_qc/scripts/compute_ld.py`:
- Filter: biallelic, MAF ≥ 0.05, ≥70% non-missing call rate
- Mixed ploidy handled (cactus haploid → dose 0/2; PanGenie diploid → 0/1/2)
- 5,000 anchor markers per run, neighbors within 1Mb
- Per pair: Pearson r² across the 231-founder dose vector

Run on **Chr1** (largest chrom, most diverse) via `run_ld_chr1.sh` SLURM job.
Outputs in `preprocess_qc/output/ld/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LD = Path('../output/ld')
SUMMARY = LD / 'ld_summary.tsv'
if not SUMMARY.exists():
    raise FileNotFoundError(f'{SUMMARY} not found — wait for aggregator job (run_aggregate_ld.sh) to land')

df = pd.read_csv(SUMMARY, sep='\t')
print(f'Loaded {len(df):,} summary rows from {SUMMARY}')
print()
print('=== pairs per (source, chrom) ===')
print(df.groupby(['source_key','chrom']).n.sum().to_string())


## 1. SNP LD-decay — apples-to-apples between panels

In [ ]:
# SNP-SNP LD decay: GrENE-Net SNPs vs production-merged SNPs
snp_pairs = df[(df.class1 == 'SNP') & (df.class2 == 'SNP')]

# Aggregate over chroms within each source × bin (weighted-by-n median is hard; pool n's and use mean of medians weighted by n)
def agg_decay(sub):
    """Combine the per-chrom medians using count-weighted average within each bin."""
    g = sub.groupby('bin_mid').apply(lambda s: pd.Series({
        'median_r2': np.average(s.median_r2, weights=s.n) if s.n.sum() > 0 else np.nan,
        'q25_r2':    np.average(s.q25_r2,    weights=s.n) if s.n.sum() > 0 else np.nan,
        'q75_r2':    np.average(s.q75_r2,    weights=s.n) if s.n.sum() > 0 else np.nan,
        'n':         s.n.sum(),
    }), include_groups=False).reset_index().sort_values('bin_mid')
    return g

fig, ax = plt.subplots(figsize=(10, 5))
for src_key, label, color in [
    ('grenenet_snp', 'GrENE-Net SNP-only (231)', '#7f8da6'),
    ('merged_snp',   'Production merged (231) — SNPs', '#1f4e79'),
]:
    sub = snp_pairs[snp_pairs.source_key == src_key]
    if len(sub) == 0:
        print(f'  {label}: no data'); continue
    g = agg_decay(sub)
    ax.plot(g.bin_mid, g.median_r2, lw=2, label=f'{label}  (n={int(g.n.sum()):,})', color=color)
    ax.fill_between(g.bin_mid, g.q25_r2, g.q75_r2, alpha=0.18, color=color)

ax.set_xscale('log')
ax.set_xlabel('pairwise distance (bp)')
ax.set_ylabel('LD r² (count-weighted across chroms; shaded = IQR)')
ax.set_title('SNP-SNP LD decay — pooled across Chr1-Chr5, MAF≥0.05, 231 founders')
ax.set_ylim(0, 1.0)
ax.legend()
plt.tight_layout(); plt.show()


## 2. SV LD-decay vs SNP LD-decay (production panel only)

How do SVs' LD profiles compare to SNPs at the same panel + chrom?

In [ ]:
# SV anchors are in the merged_sv source; we plot SV→SNP and SV→SV separately by anchor size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
# SNP→SNP baseline from production merged
snp_snp = df[(df.source_key == 'merged_snp') & (df.class1 == 'SNP') & (df.class2 == 'SNP')]
g = agg_decay(snp_snp)
ax.plot(g.bin_mid, g.median_r2, lw=2, label='SNP↔SNP (production)', color='#1f4e79')
ax.fill_between(g.bin_mid, g.q25_r2, g.q75_r2, alpha=0.15, color='#1f4e79')

# SV→SNP per SV size class
for cls, color in [('small_sv','#d8b365'), ('medium_sv','#bf812d'), ('large_sv','#8c510a')]:
    sub = df[(df.source_key == 'merged_sv') & (df.class1 == cls) & (df.class2 == 'SNP')]
    if len(sub) == 0: continue
    g = agg_decay(sub)
    ax.plot(g.bin_mid, g.median_r2, lw=2, label=f'{cls}↔SNP', color=color)
ax.set_xscale('log'); ax.set_xlabel('distance (bp)'); ax.set_ylabel('median r²')
ax.set_title('LD decay — SV anchors vs SNP neighbors')
ax.set_ylim(0, 1); ax.legend()

ax = axes[1]
# SV-class anchor vs any SV neighbor (small_sv, medium_sv, large_sv)
for cls, color in [('small_sv','#d8b365'), ('medium_sv','#bf812d'), ('large_sv','#8c510a')]:
    sub = df[(df.source_key == 'merged_sv') & (df.class1 == cls) & df.class2.isin(['small_sv','medium_sv','large_sv'])]
    if len(sub) == 0: continue
    g = agg_decay(sub)
    ax.plot(g.bin_mid, g.median_r2, lw=2, label=f'{cls}↔any SV', color=color)
ax.set_xscale('log'); ax.set_xlabel('distance (bp)'); ax.set_ylabel('median r²')
ax.set_title('LD decay — SV anchors vs SV neighbors')
ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.show()


## 3. r² distribution at fixed distance windows

In [ ]:
# Median r² in fixed distance windows (uses summary's bin median; weighted across chroms by pair count)
windows = [(1, 1_000, '<1 kb'),
           (1_000, 10_000, '1-10 kb'),
           (10_000, 100_000, '10-100 kb'),
           (100_000, 1_000_000, '100kb-1Mb')]
rows = []
for src_key, src_label in [('grenenet_snp','GrENE-Net SNP'),
                            ('merged_snp','Production SNP'),
                            ('merged_sv','Production SV (any size)')]:
    for lo, hi, label in windows:
        if src_key == 'merged_sv':
            sub = df[(df.source_key == src_key) & df.class1.isin(['small_sv','medium_sv','large_sv']) &
                     (df.bin_mid >= lo) & (df.bin_mid < hi)]
        else:
            sub = df[(df.source_key == src_key) & (df.class1 == 'SNP') & (df.class2 == 'SNP') &
                     (df.bin_mid >= lo) & (df.bin_mid < hi)]
        if len(sub) == 0: continue
        m = float(np.average(sub.median_r2, weights=sub.n))
        rows.append({'source': src_label, 'window': label, 'median_r2': m, 'n_pairs': int(sub.n.sum())})
table = pd.DataFrame(rows)
if len(table):
    pivot = table.pivot_table(index='source', columns='window', values='median_r2').round(3)
    print('=== count-weighted median r² across Chr1-Chr5 ===')
    print(pivot[['<1 kb','1-10 kb','10-100 kb','100kb-1Mb']].to_string())
    print()
    fig, ax = plt.subplots(figsize=(10,4.5))
    pivot[['<1 kb','1-10 kb','10-100 kb','100kb-1Mb']].plot(kind='bar', ax=ax)
    ax.set_ylabel('median r² (count-weighted)')
    ax.set_title('Median r² in fixed distance windows — pooled across Chr1-Chr5')
    ax.set_ylim(0, 1)
    plt.xticks(rotation=15)
    plt.tight_layout(); plt.show()
else:
    print('no rows; check ld_summary.tsv')


## 4. Median r² as a function of size class (anchor side)

Within the 1-100 kb window, what's the typical r² we can expect with a SNP neighbor?

In [ ]:
# Per (source, anchor_class, window) median r² — full pivot
windows = [(1, 1_000, '<1 kb'),
           (1_000, 10_000, '1-10 kb'),
           (10_000, 100_000, '10-100 kb'),
           (100_000, 1_000_000, '100kb-1Mb')]
rows = []
for src_key, src_label in [('grenenet_snp','GrENE-Net SNP'),
                            ('merged_snp','Production SNP'),
                            ('merged_sv','Production SV')]:
    for lo, hi, wlabel in windows:
        for cls in ['SNP','small_indel','small_sv','medium_sv','large_sv']:
            sub = df[(df.source_key == src_key) & (df.class1 == cls) &
                     (df.bin_mid >= lo) & (df.bin_mid < hi)]
            if sub.n.sum() < 10: continue
            m = float(np.average(sub.median_r2, weights=sub.n))
            rows.append({'source': src_label, 'window': wlabel,
                         'anchor_class': cls, 'n': int(sub.n.sum()),
                         'median_r2': round(m, 3)})
table = pd.DataFrame(rows)
if len(table):
    pivot = table.pivot_table(index=['source','anchor_class'], columns='window', values='median_r2',
                              aggfunc='first')[['<1 kb','1-10 kb','10-100 kb','100kb-1Mb']]
    print(pivot)
else:
    print('no rows; check ld_summary.tsv')


## 5. Headline summary

Pooled across Chr1-Chr5 (5,000 anchors per chr × source) — count-weighted medians.

Key questions:

- **Do GrENE-Net and production SNP-LD curves overlay?** If yes (curves match within IQR), the two SNP catalogs capture the same haplotype structure — sanity check passes.
- **Does SV-LD decay match SNP-LD decay at comparable distance?** If comparable → SVs are taggable by SNPs (good for hapFIRE-style LD-borrowing methods). If much lower → SVs need direct genotyping (which is exactly what we built with PanGenie).
- **Does SV-SV LD reveal clustered SV regions?** Tight SV-SV LD at short distance hints at structural-variant hotspots / TE clusters.

If you want finer per-class breakdowns, re-run `ld_one.sh` with `--n-anchors 20000` per chr.
